# Build the FaceForensics++ dataset

Downloads the FaceForensics++ c23 videos (the 1,000 originals and the four manipulation methods of the paper: Deepfakes, Face2Face, FaceSwap, NeuralTextures), crops their faces with the same RetinaFace pipeline as `build_dataset.ipynb`, and uploads the crops as a **private** Hugging Face dataset. The official train/val/test splits ship with this package (`face_decision_lab.sources.faceforensics`), so every protocol uses them.

**Access:** FaceForensics++ is shared per researcher through the authors' form, for research use only and without redistribution. The download script URL arrives by email after approval; store it in a Colab secret named `FFPP_DOWNLOAD_URL`, so it never enters this repository. Running the script means accepting the FaceForensics terms of use, which the form already asked you to accept.

**Runtime:** Colab Pro with a GPU. **Needs** the Colab secrets `FFPP_DOWNLOAD_URL` and `HF_TOKEN` (write access).

### Configuration

In [ ]:
REPO_SOURCE = "git+https://github.com/paulohenriquevn/face-decision-lab@workspace"
HF_DATASET = "paulohenriquevn/face-decision-ffpp"  # created private on upload
DOWNLOAD_DIR = "/content/ffpp"
BUILD_DIR = "/content/build_ffpp"
DATASETS = ("original", "Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures")
SERVER = "EU2"  # the only server the authors list as available in October 2026
FRAMES_PER_VIDEO = 16

### Installation

In [ ]:
%%capture
!pip install -q "face-decision-lab[build] @ {REPO_SOURCE}"
!pip uninstall -y -q onnxruntime
!pip install -q onnxruntime-gpu

In [ ]:
import os
os.environ["DEEPFACE_BACKEND_ENGINE"] = "onnx"

import onnxruntime as ort
providers = ort.get_available_providers()
print("ONNX Runtime providers:", providers)
assert "CUDAExecutionProvider" in providers, "onnxruntime-gpu cannot see the GPU; RetinaFace would run on the CPU"

### Download

The authors' script asks to press a key to accept the terms of use; the newline piped into it is that key press. Already downloaded files are skipped, so a rerun resumes.

In [ ]:
import subprocess, urllib.request
from google.colab import userdata

urllib.request.urlretrieve(userdata.get("FFPP_DOWNLOAD_URL"), "/content/ffpp_download.py")
for dataset in DATASETS:
    subprocess.run(["python", "/content/ffpp_download.py", DOWNLOAD_DIR, "-d", dataset, "-c", "c23", "-t", "videos", "--server", SERVER],
                   input = "\n", text = True, check = True)

### Build

Resumable: a rerun skips every video already in `processed.jsonl`.

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm
from face_decision_lab.build import DatasetBuilder, folder_files
from face_decision_lab.faces import RetinaFaceDetector
from face_decision_lab.sources import faceforensics

videos = list(folder_files(Path(DOWNLOAD_DIR), faceforensics.parse_path))
print(len(videos), "videos")
builder = DatasetBuilder(BUILD_DIR, RetinaFaceDetector(), frames_per_video = FRAMES_PER_VIDEO)
for item, path in tqdm(videos, desc = "faceforensics"):
    builder.add_video(item, path)
builder.flush()
print(len(builder.samples), "crops from", len({s.video_id for s in builder.samples}), "videos")

### Check detection outcomes

Compare the saved share per generator before uploading: a detector that drops one method more than the others biases the data.

In [ ]:
from collections import Counter

for generator in sorted(builder.report.counts):
    counts = builder.report.counts[generator]
    print(f"{generator:24s} saved {builder.report.saved_share(generator):6.1%}  {counts}")
print(Counter(faceforensics.official_split(s) for s in builder.samples))

### Upload as a private dataset

In [ ]:
from huggingface_hub import login
from face_decision_lab.hub import push_build

login(token = userdata.get("HF_TOKEN"))
print(push_build(BUILD_DIR, HF_DATASET))